# A Probing Study of Contextual Embeddings for Sarcasm Detection

## 1. Data Preparation

In [1]:
!pip install vaderSentiment spacy --quiet
!python -m spacy download en_core_web_sm --quiet

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import pandas as pd
import numpy as np
import spacy
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from google.colab import drive
import os
import json

In [5]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# Set your paths — adjust the folder name to wherever your files are
DATA_PATH  = '/content/drive/MyDrive/CP Project/'
TRAIN_FILE = DATA_PATH + 'SARC-train-balanced.csv'
TEST_FILE  = DATA_PATH + 'SARC-test-balanced.csv'
SAVE_PATH  = DATA_PATH  # we'll save all outputs here

os.makedirs(SAVE_PATH, exist_ok=True)

In [11]:
# ── 2. Load raw file ─────────────────────────────────────────
train_raw = pd.read_csv(TRAIN_FILE)

print("Columns:", train_raw.columns.tolist())
print("Shape:  ", train_raw.shape)
print()
print(train_raw.head(3).to_string())

Columns: ['label', 'comment', 'author', 'subreddit', 'score', 'ups', 'downs', 'date', 'created_utc', 'parent_comment']
Shape:   (1010826, 10)

   label                                                                                                                    comment     author subreddit  score  ups  downs     date          created_utc                                                                                                                          parent_comment
0      0                                                                                                                 NC and NH.  Trumpbart  politics      2   -1     -1  2016-10  2016-10-16 23:55:23                                                        Yeah, I get that argument. At this point, I'd prefer is she lived in NC as well.
1      0                                                 You do know west teams play against west teams more than east teams right?  Shbshb906       nba     -4   -1     -1  2016-11 

In [12]:
# ── 3. Standardise columns ───────────────────────────────────
def standardise(df):
    df = df.copy()

    rename_map = {
        'comment':   'text',
        'label':     'label',
        'subreddit': 'subreddit',
    }
    rename_map = {k: v for k, v in rename_map.items()
                  if k in df.columns}
    df = df.rename(columns=rename_map)

    cols_to_keep = [c for c in ['text', 'label', 'subreddit']
                    if c in df.columns]
    df = df[cols_to_keep]

    df = df.dropna(subset=['text', 'label'])
    df['label'] = df['label'].astype(int)
    df['text']  = df['text'].astype(str)

    return df

train_df = standardise(train_raw)

print("After standardising:")
print("Shape:", train_df.shape)
print("Label distribution:\n", train_df['label'].value_counts())
print()
print(train_df.head(3).to_string())

After standardising:
Shape: (1010771, 3)
Label distribution:
 label
0    505403
1    505368
Name: count, dtype: int64

                                                                                                                        text  label subreddit
0                                                                                                                 NC and NH.      0  politics
1                                                 You do know west teams play against west teams more than east teams right?      0       nba
2  They were underdogs earlier today, but since Gronk's announcement this afternoon, the Vegas line has moved to patriots -1      0       nfl


In [13]:
# ── 4. Stratified sample: 5,000 sarcastic + 5,000 literal ───
N_PER_CLASS = 5000
RANDOM_SEED = 42

def stratified_sample(df, label_val, n, seed=42):
    """
    Sample n rows with a given label, stratified by subreddit.
    Falls back to simple random sample if subreddit is missing
    or a subreddit has too few rows.
    """
    subset = df[df['label'] == label_val].copy()

    if 'subreddit' not in subset.columns:
        print(f"  No subreddit column — random sampling")
        return subset.sample(n=min(n, len(subset)),
                             random_state=seed)

    sub_counts     = subset['subreddit'].value_counts()
    sub_props      = sub_counts / sub_counts.sum()
    sub_allocs     = (sub_props * n).astype(int)

    print(f"  label={label_val}: {len(subset)} rows across "
          f"{len(sub_counts)} subreddits")

    samples = []
    for sub, alloc in sub_allocs.items():
        if alloc == 0:
            continue
        sub_df = subset[subset['subreddit'] == sub]
        alloc  = min(alloc, len(sub_df))
        samples.append(
            sub_df.sample(n=alloc, random_state=seed)
        )

    sampled = pd.concat(samples, ignore_index=True)

    # Top up if rounding left us short
    if len(sampled) < n:
        already_idx = set(sampled.index)
        remaining   = subset[~subset.index.isin(already_idx)]
        shortfall   = n - len(sampled)
        if len(remaining) >= shortfall:
            extra   = remaining.sample(n=shortfall,
                                       random_state=seed)
            sampled = pd.concat([sampled, extra],
                                ignore_index=True)

    # Trim to exactly n
    return sampled.sample(
        n=min(n, len(sampled)), random_state=seed
    )


print("Sampling sarcastic sentences...")
sarcastic_sample = stratified_sample(
    train_df, 1, N_PER_CLASS, RANDOM_SEED
)

print("Sampling literal sentences...")
literal_sample = stratified_sample(
    train_df, 0, N_PER_CLASS, RANDOM_SEED
)

# Combine and shuffle
dataset = pd.concat(
    [sarcastic_sample, literal_sample], ignore_index=True
)
dataset = dataset.sample(
    frac=1, random_state=RANDOM_SEED
).reset_index(drop=True)

print()
print("Final dataset shape:", dataset.shape)
print("Label distribution:\n", dataset['label'].value_counts())
if 'subreddit' in dataset.columns:
    print("\nSubreddit spread (top 10):\n",
          dataset['subreddit'].value_counts().head(10))

Sampling sarcastic sentences...
  label=1: 505368 rows across 8993 subreddits
Sampling literal sentences...
  label=0: 505403 rows across 12836 subreddits

Final dataset shape: (10000, 3)
Label distribution:
 label
0    5000
1    5000
Name: count, dtype: int64

Subreddit spread (top 10):
 subreddit
AskReddit          792
politics           472
worldnews          328
leagueoflegends    243
pcmasterrace       232
funny              228
pics               208
news               193
GlobalOffensive    172
nba                170
Name: count, dtype: int64


In [14]:
# ── 5. Identify sarcasm-critical tokens ──────────────────────
nlp      = spacy.load("en_core_web_sm")
analyzer = SentimentIntensityAnalyzer()

def get_critical_word(text):
    """
    Finds the sarcasm-critical token in a sentence.

    Primary method: among all content words (ADJ, ADV, NOUN,
    VERB), pick the one whose individual VADER sentiment most
    contradicts the overall sentence sentiment.

    incongruity = word_sentiment - sentence_sentiment

    High score = word is more positive than the sentence
    overall = likely the irony trigger.

    Fallback 1: most positive ADJ or ADV in the sentence.
    Fallback 2: most positive any content word.

    Returns:
        critical_word  : token text (str) or None
        critical_index : token position in sentence (int) or None
        critical_pos   : POS tag (str) or None
        incongruity    : incongruity score (float)
        method         : which method was used (str)
    """
    text = str(text).strip()
    if not text:
        return None, None, None, 0.0, 'empty'

    # Overall sentence sentiment
    sent_score = analyzer.polarity_scores(text)['compound']

    doc = nlp(text)

    # ── Primary: incongruity score ───────────────────────────
    best = {
        'word': None, 'idx': None, 'pos': None,
        'score': -999, 'method': 'incongruity'
    }

    for token in doc:
        if token.pos_ not in ['ADJ', 'ADV', 'NOUN', 'VERB']:
            continue
        if token.is_stop or token.is_punct:
            continue
        word_score  = analyzer.polarity_scores(
            token.text)['compound']
        incongruity = word_score - sent_score
        if incongruity > best['score']:
            best = {
                'word':   token.text,
                'idx':    token.i,
                'pos':    token.pos_,
                'score':  incongruity,
                'method': 'incongruity'
            }

    # ── Fallback 1: most positive ADJ / ADV ─────────────────
    if best['word'] is None or best['score'] <= 0:
        adj_adv = [
            (t.text, t.i, t.pos_,
             analyzer.polarity_scores(t.text)['compound'])
            for t in doc
            if t.pos_ in ['ADJ', 'ADV']
            and not t.is_stop and not t.is_punct
        ]
        if adj_adv:
            adj_adv.sort(key=lambda x: x[3], reverse=True)
            w, i, pos, sc = adj_adv[0]
            best = {
                'word': w, 'idx': i, 'pos': pos,
                'score': sc, 'method': 'fallback_adj_adv'
            }

    # ── Fallback 2: most positive any content word ───────────
    if best['word'] is None:
        content = [
            (t.text, t.i, t.pos_,
             analyzer.polarity_scores(t.text)['compound'])
            for t in doc
            if t.pos_ in ['ADJ', 'ADV', 'NOUN', 'VERB']
            and not t.is_stop and not t.is_punct
        ]
        if content:
            content.sort(key=lambda x: x[3], reverse=True)
            w, i, pos, sc = content[0]
            best = {
                'word': w, 'idx': i, 'pos': pos,
                'score': sc, 'method': 'fallback_any_content'
            }

    return (
        best['word'], best['idx'], best['pos'],
        best['score'], best['method']
    )


# Apply to sarcastic sentences only
print("Identifying sarcasm-critical tokens...")
print("Running spaCy + VADER on 5,000 sentences — ~3-5 min\n")

sarcastic_only = dataset[dataset['label'] == 1]\
                        .copy().reset_index(drop=True)

results = sarcastic_only['text'].apply(get_critical_word)

sarcastic_only['critical_word']  = results.apply(lambda x: x[0])
sarcastic_only['critical_index'] = results.apply(lambda x: x[1])
sarcastic_only['critical_pos']   = results.apply(lambda x: x[2])
sarcastic_only['incongruity']    = results.apply(lambda x: x[3])
sarcastic_only['method']         = results.apply(lambda x: x[4])

Identifying sarcasm-critical tokens...
Running spaCy + VADER on 5,000 sentences — ~3-5 min



In [15]:
# ── 6. Sanity checks ─────────────────────────────────────────
print("Method distribution:")
print(sarcastic_only['method'].value_counts())
print()

coverage = sarcastic_only['critical_word'].notna().sum()
print(f"Critical word found: {coverage} / "
      f"{len(sarcastic_only)} "
      f"({coverage/len(sarcastic_only)*100:.1f}%)")
print()

print("Top 20 most common critical words:")
print(sarcastic_only['critical_word'].value_counts().head(20))
print()

print("POS distribution of critical words:")
print(sarcastic_only['critical_pos'].value_counts())
print()

print("Incongruity score stats:")
print(sarcastic_only['incongruity'].describe().round(3))

Method distribution:
method
incongruity         3182
fallback_adj_adv    1818
Name: count, dtype: int64

Critical word found: 4787 / 5000 (95.7%)

Top 20 most common critical words:
critical_word
good          100
better         64
sure           62
great          60
forgot         50
right          43
want           39
know           39
obviously      38
think          37
best           37
totally        36
Maybe          34
pretty         32
clearly        29
free           27
definitely     27
Good           26
mean           25
true           24
Name: count, dtype: int64

POS distribution of critical words:
critical_pos
ADJ     1711
VERB    1244
NOUN    1204
ADV      628
Name: count, dtype: int64

Incongruity score stats:
count    5000.000
mean      -42.362
std       201.813
min      -999.000
25%         0.000
50%         0.000
75%         0.440
max         1.497
Name: incongruity, dtype: float64


In [16]:
# ── 7. Spot check — read through these manually ──────────────
print("=" * 65)
print("SPOT CHECK — do these critical words look right?")
print("=" * 65)

spot = sarcastic_only[
    sarcastic_only['critical_word'].notna()
].sample(20, random_state=42)

for _, row in spot.iterrows():
    text = str(row['text'])
    word = str(row['critical_word'])
    highlighted = text.replace(word, f"[[ {word} ]]", 1)
    print(f"→ {highlighted}")
    print(f"  Word: '{word}' | "
          f"POS: {row['critical_pos']} | "
          f"Incongruity: {row['incongruity']:.3f} | "
          f"Method: {row['method']}")
    print()

SPOT CHECK — do these critical words look right?
→ didnt u [[ know ]] u need a penis to ghost-bust?
  Word: 'know' | POS: VERB | Incongruity: 0.000 | Method: incongruity

→ Turns out [[ old ]] Ronny was an Anarchist
  Word: 'old' | POS: ADJ | Incongruity: 0.000 | Method: fallback_adj_adv

→ Because Reddit never downvotes anything constructive and/or [[ useful ]].
  Word: 'useful' | POS: ADJ | Incongruity: 0.440 | Method: fallback_adj_adv

→ Ah yes, the original and [[ hilarious ]] attack helicopter joke
  Word: 'hilarious' | POS: ADJ | Incongruity: 0.402 | Method: fallback_adj_adv

→ This must have been [[ fun ]] for passengers in the terminal to watch as they were boarding their flights.
  Word: 'fun' | POS: ADJ | Incongruity: 0.511 | Method: fallback_adj_adv

→ Pfft, big deal, [[ pretty ]] sure the Pats have released and re-signed Gronk's little brother at least twice that many times
  Word: 'pretty' | POS: ADV | Incongruity: 0.494 | Method: fallback_adj_adv

→ Yeah, let's [[ celebra

In [17]:
def get_critical_word_strict(text):
    """
    Stricter version — only returns a result when confident.

    Conditions for a valid critical word:
    1. Must be ADJ or ADV only (not NOUN or VERB)
    2. Word VADER score must be clearly positive (> 0.3)
    3. Sentence VADER score must be clearly negative (< -0.1)
       OR sentence is neutral but word is strongly positive (> 0.5)
    4. Incongruity score must be > 0.3

    Returns None if no confident candidate found.
    """
    text = str(text).strip()
    if not text:
        return None, None, None, 0.0, 'empty'

    sent_score = analyzer.polarity_scores(text)['compound']
    doc = nlp(text)

    candidates = []
    for token in doc:
        if token.pos_ not in ['ADJ', 'ADV']:
            continue
        if token.is_stop or token.is_punct:
            continue
        if len(token.text) < 3:  # skip very short words
            continue

        word_score  = analyzer.polarity_scores(token.text)['compound']
        incongruity = word_score - sent_score

        # Must meet all three conditions
        word_positive    = word_score > 0.3
        sentence_not_pos = sent_score < 0.1
        strong_incongruity = incongruity > 0.3

        if word_positive and sentence_not_pos and strong_incongruity:
            candidates.append((
                token.text, token.i, token.pos_,
                incongruity, 'strict'
            ))

    if not candidates:
        return None, None, None, 0.0, 'no_confident_candidate'

    # Pick the one with highest incongruity
    candidates.sort(key=lambda x: x[3], reverse=True)
    w, i, pos, sc, method = candidates[0]
    return w, i, pos, sc, method

In [18]:
print("Applying strict critical word detection...")
results_strict = sarcastic_only['text'].apply(get_critical_word_strict)

sarcastic_only['critical_word']  = results_strict.apply(lambda x: x[0])
sarcastic_only['critical_index'] = results_strict.apply(lambda x: x[1])
sarcastic_only['critical_pos']   = results_strict.apply(lambda x: x[2])
sarcastic_only['incongruity']    = results_strict.apply(lambda x: x[3])
sarcastic_only['method']         = results_strict.apply(lambda x: x[4])

# How many did we get?
found = sarcastic_only['critical_word'].notna().sum()
print(f"Critical word found (strict): {found} / {len(sarcastic_only)} "
      f"({found/len(sarcastic_only)*100:.1f}%)")

# Spot check again
print("\nSPOT CHECK — strict version:")
print("=" * 65)
spot = sarcastic_only[
    sarcastic_only['critical_word'].notna()
].sample(20, random_state=42)

for _, row in spot.iterrows():
    text = str(row['text'])
    word = str(row['critical_word'])
    highlighted = text.replace(word, f"[[ {word} ]]", 1)
    print(f"→ {highlighted}")
    print(f"  Word: '{word}' | "
          f"Incongruity: {row['incongruity']:.3f}")
    print()

Applying strict critical word detection...
Critical word found (strict): 185 / 5000 (3.7%)

SPOT CHECK — strict version:
→ Quality of life behind the Iron Curtain was really outpacing the West but no one knew it because [[ powerful ]] people spent a lot of money to hide that fact.
  Word: 'powerful' | Incongruity: 0.460

→ Oh [[ sure ]] lets just fire tear gas into a crowd of pissed protesters, totally will be fine.
  Word: 'sure' | Incongruity: 0.948

→ I wonder if there is a [[ positive ]] correlation between poverty and poor education.
  Word: 'positive' | Incongruity: 0.979

→ Well, guess ill have to kill every [[ fresh ]] spawn with cargo pants now.
  Word: 'fresh' | Incongruity: 0.943

→ Mmm nothing [[ better ]] than stubble rubbing against my, uh, stubble.
  Word: 'better' | Incongruity: 0.782

→ Hostility changes nobody's point of view,you know.And it *would* come out [[ nicely ]],thank you!
  Word: 'nicely' | Incongruity: 1.025

→ Onviously if it is [[ good ]] for alabama and 

In [19]:
# ── 8. Save to Drive ─────────────────────────────────────────

# Main dataset — all 10,000 sentences
dataset_path = SAVE_PATH + 'dataset_10k.csv'
dataset.to_csv(dataset_path, index=False)
print(f"Saved: {dataset_path}  "
      f"({os.path.getsize(dataset_path)/1024:.0f} KB)")

# Sarcastic sentences with critical word annotations
critical_path = SAVE_PATH + 'sarcastic_with_critical_words.csv'
sarcastic_only.to_csv(critical_path, index=False)
print(f"Saved: {critical_path}  "
      f"({os.path.getsize(critical_path)/1024:.0f} KB)")

# Summary log
summary = {
    'total_sentences':        int(len(dataset)),
    'sarcastic':              int((dataset['label']==1).sum()),
    'literal':                int((dataset['label']==0).sum()),
    'random_seed':            RANDOM_SEED,
    'source':                 'train-balanced.csv only',
    'critical_word_coverage': f"{coverage}/{len(sarcastic_only)}",
    'method_distribution':    sarcastic_only['method']
                              .value_counts().to_dict(),
}

summary_path = SAVE_PATH + 'phase1_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"Saved: {summary_path}")

print("\nPhase 1 complete.")

Saved: /content/drive/MyDrive/CP Project/dataset_10k.csv  (682 KB)
Saved: /content/drive/MyDrive/CP Project/sarcastic_with_critical_words.csv  (490 KB)
Saved: /content/drive/MyDrive/CP Project/phase1_summary.json

Phase 1 complete.
